> 在线讲义：[第 33 章 · 上下文、流水线、专家并行：切位置、切层、切专家](https://wikieden.github.io/ai-infra-from-constraints/lessons/33-cp-pp-ep.html) · 本书《从约束推导 AI Infra》

# 第 33 章 · 上下文、流水线、专家并行：切位置、切层、切专家

配套讲义：仓库里 `ai-infra/notes/lessons/33-cp-pp-ep.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



因果配对的负载、跨卡合并注意力统计量、式 (6-7) 的流水、专家 dispatch 字节取自 AII 第 6 章 §6.2.4–§6.2.6。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
KiB, MiB, GiB = 2**10, 2**20, 2**30

**函数 `causal_pairs`**：因果注意力：位置 i（从 1 起）的查询看 i 个键。

In [ ]:
def causal_pairs(positions: list[int]) -> int:
    """因果注意力：位置 i（从 1 起）的查询看 i 个键。"""
    return sum(positions)

**函数 `merge_stats`**：各卡 (m_r, l_r, u_r) 合并成最终输出 u / l（标量值版本）。

In [ ]:
def merge_stats(parts: list[tuple[float, float, float]]) -> float:
    """各卡 (m_r, l_r, u_r) 合并成最终输出 u / l（标量值版本）。"""
    m = max(p[0] for p in parts)
    l = sum(math.exp(mr - m) * lr for mr, lr, _ in parts)
    u = sum(math.exp(mr - m) * ur for mr, _, ur in parts)
    return u / l

**函数 `local_stats`**：小工具。

In [ ]:
def local_stats(scores: list[float], values: list[float]) -> tuple[float, float, float]:
    m = max(scores)
    w = [math.exp(s - m) for s in scores]
    return m, sum(w), sum(wi * v for wi, v in zip(w, values))

**函数 `pipeline`**：式 (6-7)：(总时间, 利用率)。

In [ ]:
def pipeline(q: int, b: int, t: float = 1.0) -> tuple[float, float]:
    """式 (6-7)：(总时间, 利用率)。"""
    return (q + b - 1) * t, b / (q + b - 1)

**函数 `pipeline_uneven`**：各阶段耗时不等时每个 micro-batch 的完成时刻（无限缓冲）。

In [ ]:
def pipeline_uneven(stages: list[float], b: int) -> list[float]:
    """各阶段耗时不等时每个 micro-batch 的完成时刻（无限缓冲）。"""
    free = [0.0] * len(stages)
    done = []
    for _ in range(b):
        t = 0.0
        for s, d in enumerate(stages):
            t = max(t, free[s]) + d
            free[s] = t
        done.append(t)
    return done

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：CP，8 个位置的因果注意力")
print(f"  共 {causal_pairs(list(range(1, 9)))} 对；连续切半 {causal_pairs([1, 2, 3, 4])} / {causal_pairs([5, 6, 7, 8])}；"
      f"首尾搭配 {causal_pairs([1, 2, 7, 8])} / {causal_pairs([3, 4, 5, 6])}")
s, v = [0.5, 2.0, -1.0, 1.5], [1, 2, 3, 4]
two = merge_stats([local_stats(s[:2], v[:2]), local_stats(s[2:], v[2:])])
print(f"  两卡各算统计量再合并 {two:.4f}；一次算整行 {merge_stats([local_stats(s, v)]):.4f}")

In [ ]:
print("\n第二步：PP，式 (6-7)")
for b in (4, 16):
    T, eta = pipeline(4, b)
    print(f"  4 阶段、{b} 个 micro-batch：{T:.0f} ms，利用率 {eta:.0%}")
print(f"  阶段 1、1、2、1 ms，4 个 micro-batch 完成于 {pipeline_uneven([1, 1, 2, 1], 4)} ms")

In [ ]:
print("\n第三步：EP")
print(f"  8 个专家两两在 4 张远端卡：按专家发 {8 * 8} KiB，按目的卡发 {4 * 8} KiB")
e = 3 * 4096 * 1536 * 2
print(f"  每专家 {e // MiB} MiB，单层 128 个 {128 * e / GiB:.1f} GiB，94 层 {94 * 128 * e / GiB:.0f} GiB，八卡每卡 {94 * 128 * e / GiB / 8:.1f} GiB")

## 3. 代码里才看得到的

1. `pipeline_uneven` 假设缓冲无限：慢阶段前的积压不会让上游停；有限缓冲时上游也会被拖慢。
2. 测试把 6 个分数切成 1、2、3 段分别合并，结果都和整行相同。
3. `local_stats` 返回的 $u$ 是标量值的加权和；真实注意力里 $u$ 是一个 $d$ 维向量，合并方式一样。

## 4. 自己改一改

- 给 `pipeline_uneven` 加缓冲容量参数，看 1、1、2、1 ms 时前两阶段何时被迫暂停。
- 写一个函数，为 $n$ 个位置、$k$ 张卡找最均衡的分组。
- 把 `merge_stats` 改成向量版，验证与 `softmax(s) @ V` 一致。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 分读小说，后半本要知道前情；前少后多，首尾搭配才匀；
> 各卡统计量合并，不能各算完再平均；
> 洗烘叠包，第一筐要走四道，筐多机器才不闲；最慢那道定节拍；
> 大厅挂号、科室看病、报告送回；同一科室只跑一趟。